# Evaluation

Jusqu'à présent, nous avons vu le strict minimum à réaliser pour un RAG mais il existe une multitude d'embedder, reranker, de façons de traiter et extraire les données. Avant de pouvoir aller plus, il faut des métriques qui nous permettent de mesurer l'efficacité de chacune des méthodes.
Il faut également au préalable consituer un jeu de données avec une paire de questions réponses attendues construite pas les personnes terrain. C'est ce qu'on appelle le *golden dataset*. Comptez ~50 questions de difficulté variable qui doivent couvrir au maximum le sujet traité.

Dans un RAG, il y a deux éléments à suivre:
1. La récupération. Recupère-t-on le bon contexte, les bons documents et dans le bon ordre ?
2. La génération. La réponse du LLM est-elle correcte, fidèle au contexte founrni ? Pas d'hallucination ? 

A noter qu'ici dans le golden dataset, on a mis des chunks à récupérer mais les personnes métiers metteront des documents et peut-être dans certains cas des passages mais qui ne correspondent pas forcément à notre découpe.

```sh
uv add "deepeval>=4.3" "transformers<5.18"
```

## Mesures sur la récupération

Les mesures sur la récupération sont plus simples à comprendre et à calculer.

### Hit Rate@k

C'est la métrique la plus simple et la plus parlante. Elle permet de déterminer si la réponse est dans les k premiers chunks récupérés par la recherche. Elle ne dit rien sur la qualité des autres chunks ni sur l'ordre dans lequel on a récupéré la réponse. Le hit rate ne permet pas non plus de savoir si toutes les chunks attendus sont récupérés.

Elle permet de mesurer si la recherche sémantique et lexicale retrouve les chunks attendus

In [30]:
import os
import numpy as np
import polars as pl

import itertools

from openai import OpenAI, AsyncOpenAI
from deepeval import evaluate as de_evaluate
from deepeval.test_case import LLMTestCase
from deepeval.evaluate import AsyncConfig, DisplayConfig, ErrorConfig
from deepeval.models import LocalModel
from deepeval.metrics import (AnswerRelevancyMetric, ContextualRecallMetric, ContextualPrecisionMetric, FaithfulnessMetric, GEval)
from deepeval.test_case import SingleTurnParams

In [2]:
golden_dataset = {
    'q1': {'expected_chunks': ['c1', 'c3']},
    'q2': {'expected_chunks': ['c12', 'c8']}
}

retrieval = {
    'q1': {'retrieved_chunks': ['c5', 'c7', 'c1', 'c6', 'c12']},
    'q2': {'retrieved_chunks': ['c12', 'c4', 'c8', 'c2', 'c9']}
}

def hit(expected: list, retrieved: list, k: int):
    return any(c in expected for c in retrieved[:k])

def evaluate(qrels: list, retrieved: list, top_k: tuple[int], scorer: callable):
    results = []
    for k in top_k:
        for q in golden_dataset.keys():
            expected = golden_dataset.get(q).get('expected_chunks')
            retrieved = retrieval.get(q).get('retrieved_chunks')
            has_hit = scorer(expected, retrieved, k)
            results.append(
                {
                    'q': q,
                    'k': k,
                    'score': has_hit
                }
            )
    return results

top_k = (1, 3, 5)
results = evaluate(golden_dataset, retrieval, top_k, hit)
results = pl.from_dicts(results)
display(results)
results.group_by('k').agg(
    pl.col('score').mean()
).sort('k')

q,k,score
str,i64,bool
"""q1""",1,false
"""q2""",1,true
"""q1""",3,true
"""q2""",3,true
"""q1""",5,true
"""q2""",5,true


k,score
i64,f64
1,0.5
3,1.0
5,1.0


### Mean Reciprocal Rank (MMR)

Permet de mesurer quand arrive le premier bon chunk.
- bon chunk en première place == 1/1
- 2e place = 1/2
- 3e place == 1/3

Accord beaucoup d'importance à la première place.

Mesure surtout le reranker dont le rôle est de faire remonter les chunks les plus pertinents en haut de la liste.

In [3]:
def mmr(expected: list, retrieved: list, k: int):
    for c in expected:
        for idx, retrieved_c in enumerate(retrieved[:k], start=1):
            if c == retrieved_c:
                return 1/idx
    return 0.0

top_k = (1, 3, 5)
results = evaluate(golden_dataset, retrieval, top_k, mmr)
results = pl.from_dicts(results)
display(results)
results.group_by('k').agg(
    pl.col('score').mean()
).sort('k')

q,k,score
str,i64,f64
"""q1""",1,0.0
"""q2""",1,1.0
"""q1""",3,0.333333
"""q2""",3,1.0
"""q1""",5,0.333333
"""q2""",5,1.0


k,score
i64,f64
1,0.5
3,0.666667
5,0.666667


### Recall@k

Nos deux métriques précédentes ne prennent pas en compte le nombre de document qu'on aurait du retrouver. Qu'on récupère un des documents ou tous peut ne rien changer à nos deux métriques précédentes. Le Recall@K est donc une mesure de complétude. Ai-je bien récupérer tout ce dont j'ai besoin pour répondre ?

In [4]:
def recall(expected: list, retrieved: list, k: int):
    hit = 0
    for c in expected:
        for idx, retrieved_c in enumerate(retrieved[:k], start=1):
            if c == retrieved_c:
                hit += 1
    return hit/len(expected)

top_k = (1, 3, 5)
results = evaluate(golden_dataset, retrieval, top_k, recall)
results = pl.from_dicts(results)
display(results)
results.group_by('k').agg(
    pl.col('score').mean()
).sort('k')

q,k,score
str,i64,f64
"""q1""",1,0.0
"""q2""",1,0.5
"""q1""",3,0.5
"""q2""",3,1.0
"""q1""",5,0.5
"""q2""",5,1.0


k,score
i64,f64
1,0.25
3,0.75
5,0.75


### normalized Discounted Cumulative Gain (nDCG@k)

C'est la métrique reine des classements de recherche d'information comme [MTEB](https://huggingface.co/spaces/mteb/leaderboard) pour les embedders par exemple.

Contrairement aux métriques précédentes, elle a besoin d'un degré de pertinence(grade) associé à chaque réponse attendue. Il prend donc en compte la position(rang) d'un chunk ainsi que se pertinence. Un chunk très pertinent qui arrive en dernier pénalisera fortement notre récupération.

Le calcul est cependant un peu plus compliqué et se compose de 3 étapes:
1. Gain cumulé avec décote (DCG). La somme des contributions des chunks selon leur rang et grade
2. le DCG idéal (IDCG). Le meilleur classement possible
3. le nDCG *i.e.* DCG/IDCG

A noter qu'un grade 3 > 1. Donc un chunk avec un grade 3 compte 3x plus qu'un grade 1 qu'importe le rang.

**Gain linéaire**

$$
\mathrm{DCG}_k = \sum_{i=1}^{k} \frac{grade_i}{\log_2(rang+1)}
$$

Une formula alternative existe pour le DCG:

**Gain exponentiel**

$$
\mathrm{DCG}_k = \sum_{i=1}^{k} \frac{2^{grade_i} - 1}{\log_2(rang+1)}
$$

**nDCG**

$$
\mathrm{nDCG}_k = \frac{\mathrm{DCG}_k}{\mathrm{IDCG}_k}
$$

In [5]:
rangs = (1, 2, 3) 
grades = (1, 2, 3)
dcg = pl.DataFrame(itertools.product(rangs, grades), schema=('rang', 'grade'))
dcg = dcg.with_columns(
    dcg = pl.col('grade')/(pl.col('rang')+1).log(2)
).sort('dcg', descending=True)
dcg

rang,grade,dcg
i64,i64,f64
1,3,3.0
1,2,2.0
2,3,1.892789
3,3,1.5
2,2,1.26186
1,1,1.0
3,2,1.0
2,1,0.63093
3,1,0.5


Ici l'idéal est
- rang 1 x grade 3
- rang 2 x grade 2
- rang 3 x grade 1

In [6]:
idcg = dcg.filter(
    pl.col('rang') == pl.col('grade').rank('dense', descending=True)
)
idcg

rang,grade,dcg
i64,i64,f64
1,3,3.0
2,2,1.26186
3,1,0.5


In [ ]:
golden_dataset = {
    'q1': {'expected_chunks': {'c1': 1, 'c3': 2}},
    'q2': {'expected_chunks': {'c12': 2, 'c8': 1}},

}

retrieval = {
    'q1': {'retrieved_chunks': ['c5', 'c7', 'c1', 'c6', 'c12']},
    'q2': {'retrieved_chunks': ['c12', 'c4', 'c8', 'c2', 'c9']}
}

def ndcg(expected: dict[str, int], retrieved: list[str], k: int) -> dict[str, float]:
    dcg = sum(
        expected.get(chunk, 0) / np.log2(rank + 1)
        for rank, chunk in enumerate(retrieved[:k], start=1)
    )
    ideal_grades = sorted(expected.values(), reverse=True)[:k]
    idcg = sum(
        grade / np.log2(rank + 1)
        for rank, grade in enumerate(ideal_grades, start=1)
    )

    return {
        'dcg': dcg,
        'idcg': idcg,
        'ndcg': dcg / idcg if idcg > 0 else 0.0,
    }

top_k = (1, 3, 5)
results = evaluate(golden_dataset, retrieval, top_k, ndcg)
results = pl.from_dicts(results).unnest('score')
display(results)

results.group_by('k').agg(ndcg=pl.col('ndcg').mean()).sort('ndcg', descending=True)

q,k,dcg,idcg,ndcg
str,i64,f64,f64,f64
"""q1""",1,0.0,2.0,0.0
"""q2""",1,2.0,2.0,1.0
"""q1""",3,0.5,2.63093,0.190047
"""q2""",3,2.5,2.63093,0.950234
"""q1""",5,0.5,2.63093,0.190047
"""q2""",5,2.5,2.63093,0.950234


k,ndcg
i64,f64
5,0.570141
3,0.570141
1,0.5


Aucune métrique ne résume tout. On en suit plusieurs, et on choisit celle qui décide selon l'usage.

| Métrique | Question posée | Sensible au rang | Utilise le degré 1 | Utile surtout pour |
|---|---|---|---|---|
| **Hit rate@k** | Au moins un bout de la réponse est-elle dans le top k ? | Non | Non | Le résumé le plus parlant ; le plafond de la chaîne (k grand) |
| **MRR** | À quel rang arrive la première réponse ? | Fortement | Non | Évaluer un reranker, un top très court |
| **Recall@k** | Quelle part des informations nécessaires est retrouvée ? | Non | Non | Questions à plusieurs faits : dossier + notice, listes |
| **nDCG@k** | Le classement est-il bon dans son ensemble ? | Oui | Oui | La comparaison globale la plus fine |

## Mesures sur la qualité des réponses

Les mesures précédentes évaluent principalement la récupération mais ne permettent pas de savoir si la réponse générée par notre agent a bien utilisé les chunks ou si les réponses générées est bonne. Elles peuvent également être assez fastidieuses à calculer puisqu'il faut un travail manuel d'écriture des questions, des documents et passage à retrouver, *etc*.

Pour ce faire nous allons utiliser des LLM comme juge (*LLM As Judge*). Deepeval est une librairie spécifiquement prévue pour cette utilisation. Il est conseillé d'utiliser un autre modèle que celui utilisé pour la génération parce qu'ils ont une certaine tendance à préférer leurs propres réponses.

Pour que cela fonctionne, il nous faut quatre éléments:
1. user_input: la question
2. retrieved_contexts: les chunks
3. response: la réponse générée par notre LLM
4. reference: la réponse attendue

Deepeval propose des dizaines de métriques mais nous allons nous concentrer sur quelques unes uniquement. A noter que les deux premières sont proches de celles déjà vues mais elles sont un complément. Par exemple le Recall@K permet de savoir si on a bien récupérer les chunks/documents attendus mais on ne sait pas s'ils sont utilisés par le LLM dans sa réponse.

| Métrique | Niveau | Question posée | Ce que le juge compare | Référence nécessaire |
|---|---|---|---|---|
| **Contextual precision** | Retrieval | Les passages utiles sont-ils en tête ? | Chaque chunk, à la réponse attendue | Oui |
| **Contextual recall** | Retrieval | Les passages contiennent-ils tout ce qu'il faut ? | La réponse attendue, aux chunks | Oui |
| **Faithfulness** | Génération | La réponse est-elle fidèle aux passages ? | La réponse, aux chunks | Non |
| **Answer relevancy** | Génération | La réponse traite-t-elle la question posée ? | La réponse, à la question | Non |
| **Correctness** (G-Eval) | Génération | La réponse est-elle juste ? | La réponse, à la réponse attendue | Oui |


In [ ]:
## Chunks inutiles
q1 = {
    'name': 'q1',
    'input': "Est-ce que je peux mettre de l'huile de colza (moins cher que le diesel) à la place du diesel dans ma peugot 208 tdi de 2008 ?",
    'retrieval_context': [
        "La peugeot 208 a été une grande réussite et été vendue à plus de 200000 exemplaires.",
        "Un moteur diesel est plus intéressant si vous roulez au moins 25000km/an.",
        "L'huile de colza est une bonne alternative moins onéreuse à l'huile d'olive pour la cuisson des patates rissolées."
    ],
    'actual_output': "Non vous ne pouvez pas faire des patates rissolées avec du diesel.",
    'expected_output': "Sur ces modèles de peugeot, vous pouvez en effet remplacez le diesel par de l'huile de colza"
}

## Chunks utiles mais répond à côté de la plaque
q2 = {
    "name": "q2",
    "input": "Combien de temps faut-il faire cuire des gnocchis frais ?",
    "retrieval_context": [
        "En Argentine, on mange des gnocchis le 29 de chaque mois : la tradition veut qu'on glisse "
        "un billet sous son assiette pour attirer la prospérité.",
        "Gnocchis frais : comptez 2 à 3 minutes de cuisson dans une grande casserole d'eau bouillante salée.",
        "Les gnocchis sont cuits dès qu'ils remontent à la surface : retirez-les avec une écumoire.",
    ],
    "actual_output": "En Argentine, on mange des gnocchis le 29 de chaque mois, avec un billet sous "
                     "l'assiette pour attirer la prospérité.",
    "expected_output": "2 à 3 minutes dans de l'eau bouillante salée : ils sont cuits dès qu'ils "
                       "remontent à la surface.",
}

## Bons chunks, dans le bon ordre mais mauvaise réponse
q3 = {
    "name": "q3",
    "input": "Est-ce que je peux donner un carré de chocolat à mon chien ?",
    "retrieval_context": [
        "Le chocolat contient de la théobromine, toxique pour les chiens : même une petite quantité "
        "peut provoquer des vomissements, de l'agitation et des troubles cardiaques.",
        "Si votre chien a mangé du chocolat, contactez rapidement un vétérinaire.",
        "Le cacao est cultivé principalement en Côte d'Ivoire et au Ghana.",
    ],
    "actual_output": "Oui, sans problème : un carré de chocolat de temps en temps fait une excellente "
                     "récompense pour votre chien.",
    "expected_output": "Non : le chocolat contient de la théobromine, toxique pour les chiens, même en "
                       "petite quantité. S'il en a mangé, contactez un vétérinaire.",
}

test_cases = [q1, q2, q3]

On demande un json_object en sortie, bien plus simple à manipuler. Ici, on ne passe pas de template mais c'est souhaitable.

LocalModel par défaut impose une température de 0 au modèle afin d'obtenir un jugement stable si on fait plusieurs expériences. 

LocalModel assure également les appels asynchrones afin de pouvoir traiter plusieurs paires de questions/réponses en parallèle.

```sh
llama-server \
-m ~/models/qwen3.8-27b/llama/Qwen3.8-27B-UD-Q4_K_XL.gguf \
-np 4 -c 100000 -ngl 99 \
-b 2048 -ub 512 \
-ctk q8_0 -ctv q8_0 \
--spec-type draft-mtp --spec-draft-n-max 2 \
--port 9931 --host 0.0.0.0 --alias qwen38
```

In [ ]:
JUDGE_URL = 'http://100.85.184.14:9931/v1'
JUDGE_NAME = 'qwen38'

judge = LocalModel(model=JUDGE_NAME, base_url=JUDGE_URL
                   , api_key='no-sk'
                   , generation_kwargs={'response_format': {'type': 'json_object'}}
                   )

``penalize_ambiguous_claims`` permet de pénaliser les passages qui ne sont pas confirmés ou infirmés par les chunks. On ne sait pas si c'est vrai ou non et cela pourrait être une hallucination. Dans un RAG, on cherche très régulièrement à contrer ce genre de phénomènes.

DeepEval permet également de créer ses propres métriques en passant à notre juges les éléments à prendre en compte dans son jugement et comment procéder pour son analyse. Ceci nous permet de créer des métriques propre à notre domaine.

Le terme de GEval provient d'un [article](https://arxiv.org/abs/2303.16634) publié par microsoft sur les mesures d'alignement des LLM et a été repris par tout le monde depuis.

In [10]:
contextual_precission = ContextualPrecisionMetric(model=judge)
contextual_recall = ContextualRecallMetric(model=judge) 
faithfullness = FaithfulnessMetric(model=judge, penalize_ambiguous_claims=True)
answser_relevancy = AnswerRelevancyMetric(model=judge)

correcteness = GEval(
    name = 'correctness',
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.EXPECTED_OUTPUT],
    evaluation_steps=[
        "Compare the facts of the actual output and the expected output.",
        "Penalize any fact that contradicts the expected output."
        "Missing facts in the actual output should also be penalized",
    ],
    model=judge
)

METRICS = [contextual_precission, contextual_recall, faithfullness, answser_relevancy, correcteness]

In [ ]:
results = []
for q in test_cases:
    for metric in METRICS:
        test_case = LLMTestCase(
            input=q.get('input'),
            retrieval_context=q.get('retrieval_context'),
            actual_output=q.get('actual_output'),
            expected_output=q.get('expected_output')
            )
        results.append({
            'q': q.get('name'),
            'metric': metric.__name__,
            'score': metric.score,
            'reason': metric.reason,
        })
results = pl.from_dicts(results)
with pl.Config(fmt_str_lengths=200):
    display(results)



q,metric,score,reason
str,str,f64,str
"""q1""","""Contextual Precision""",1.0,"""The score is 1.00 because the first and second retrieval context nodes are highly relevant, noting 'Le chocolat contient de la théobromine, toxique pour les chiens' and 'Si votre chien a mangé du choc…"
"""q1""","""Contextual Recall""",1.0,"""The score is 1.00 because every sentence in the expected output is well supported by node(s) in retrieval context: sentence 1 by node 1 and sentence 2 by node 2, so the recall is spot-on. Nice work!"""
"""q1""","""Faithfulness""",0.0,"""The score is 0.00 because the actual output contradicts the retrieval context by implying chocolate is a safe or excellent reward for a dog, while the context states chocolate contains toxic theobromi…"
"""q1""","""Answer Relevancy""",1.0,"""The score is 1.00 because the answer directly addresses the question about giving a chocolate square to a dog, with no irrelevant statements—nice and on point!"""
"""q1""","""correctness [GEval]""",0.0,"""The actual output directly contradicts the expected output by saying a square of chocolate is safe for a dog, while the expected output states chocolate is toxic due to theobromine and advises contact…"
…,…,…,…
"""q3""","""Contextual Precision""",1.0,"""The score is 1.00 because the first and second retrieval context nodes are highly relevant, noting 'Le chocolat contient de la théobromine, toxique pour les chiens' and 'Si votre chien a mangé du choc…"
"""q3""","""Contextual Recall""",1.0,"""The score is 1.00 because every sentence in the expected output is well supported by node(s) in retrieval context: sentence 1 by node 1 and sentence 2 by node 2, so the recall is spot-on. Nice work!"""
"""q3""","""Faithfulness""",0.0,"""The score is 0.00 because the actual output contradicts the retrieval context by implying chocolate is a safe or excellent reward for a dog, while the context states chocolate contains toxic theobromi…"


La fonction ``evaluate`` de DeepEval permet de s'épargner ces boucles.

In [35]:
test_cases = [LLMTestCase(**q) for q in test_cases]

results = de_evaluate(
    test_cases,
    METRICS,
    async_config = AsyncConfig(max_concurrent=4), # SI np == 4 dans llama-server
    display_config = DisplayConfig(),
    error_config = ErrorConfig(ignore_errors=True),
)

✨ You're running DeepEval's latest Contextual Precision Metric! (using qwen38 (Local Model), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using qwen38 (Local Model), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using qwen38 (Local Model), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using qwen38 (Local Model), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest correctness [GEval] Metric! (using qwen38 (Local Model), strict=False, 
async_mode=True)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ q1                                                                                                          │
│  ├──   Input:              Est-ce que je peux mettre de l'huile de colza (moins cher que le diesel) à la        │
│  │                         place du diesel dans ma peugot 208 tdi de 2008 ?                                     │
│  │     Actual Output:      Non vous ne pouvez pas faire des patates rissolées avec du diesel.                   │
│  │     Expected Output:    Sur ces modèles de peugeot, vous pouvez en effet remplacez le diesel par de          │
│  │                         l'huile de colza                                                                     │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric               ┃ Score ┃ Threshold ┃ Reason                                                │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Contextual Precision │ 0.00  │ 0.50      │ The score is 0.00 because all three retrieval         │
│              │                      │       │           │ context nodes are irrelevant to the input. The        │
│              │                      │       │           │ first node only says "La peugeot 208 a été une        │
│              │                      │       │           │ grande réussite et été vendue à plus de 200000        │
│              │                      │       │           │ exemplaires," which does not address using rapeseed   │
│              │                      │       │           │ oil as fuel. The second node says "Un moteur diesel   │
│              │                      │       │           │ est plus intéressant si vous roulez au moins          │
│              │                      │       │           │ 25000km/an," which is about diesel engine usage,      │
│              │                      │       │           │ not replacing diesel with rapeseed oil. The third     │
│              │                      │       │           │ node mentions "L'huile de colza est une bonne         │
│              │                      │       │           │ alternative moins onéreuse à l'huile d'olive pour     │
│              │                      │       │           │ la cuisson des patates rissolées," but this is        │
│              │                      │       │           │ about cooking, not vehicle fuel. Since no relevant    │
│              │                      │       │           │ node is ranked above irrelevant nodes, the score      │
│              │                      │       │           │ cannot be higher.                                     │
│        FAIL  │ Contextual Recall    │ 0.00  │ 0.50      │ The score is 0.00 because sentence 1 claims Peugeot   │
│              │                      │       │           │ models can replace diesel with rapeseed oil, but no   │
│              │                      │       │           │ node(s) in retrieval context support this; node 2     │
│              │                      │       │           │ only mentions “moteur diesel” and node 3 only         │
│              │                      │       │           │ associates “L'huile de colza” with cooking.           │
│        FAIL  │ Faithfulness         │ 0.00  │ 0.50      

⚠ WARNING: No hyperparameters logged.
» ]8;id=423749;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 127.6s | token cost: None)
» Test Results (3 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 3

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.